Проверим, что файл рабочий и выведем первые 5 строк

In [1]:
import csv

with open("website_visits.csv", 'r') as csv_file:
    file = csv.reader(csv_file, delimiter=';')
    i = 5
    for row in file:
        if i > 0:
            print(row)
            i -= 1
        else:
            break

['https://gonzales-bautista.com/', '2024-05-28 05:58:33.853479']
['https://gonzales-bautista.com/', '2024-05-26 03:44:36.853479']
['https://www.davis-berg.com/', '2024-05-31 17:22:01.853479']
['https://www.montoya.com/', '2024-05-29 13:41:35.853479']
['https://gonzales-bautista.com/', '2024-06-01 06:49:16.853479']


За один таск MapReduce невозможно одновременно и отсортировать все данные, и вывести топ k элементов за O(1). По этой причине будем использовать 2 таска:
1-й MapReduce: Группируем все сайты по дате обращения и сайту.
2-й MapReduce: Map переформатирует данные, чтобы в 1-м столбце была дата, а во 2-м - кол-во обращений. После этого данные будут отсортированы через составной ключ, состоящий из даты и кол-ва обращений. Благодаря этому получиться корректно подготовить данные к Reduce. Reduce будет проходиться по каждой дате и выводить по 5 сайтов, а остальные отбрасывать.

### 1. Реализация шага "Map_1"

In [2]:
%%writefile mapper_1.py

import sys


def mapper():
    # строки с названиями столбцов нет, поэтому предобработки нет
    # строка имеет следующий вид: {site};{date}\n
    for line in sys.stdin:
        site, time = line.strip().split(';')
        date = time.split()[0]
        if site != "" and date != "":
            print(f"{date};{site}\t1")
        else:
            # логирование
            pass


if __name__ == "__main__":
    mapper()

Writing mapper_1.py


### 2. Реализация шага "Reduce_1"

In [3]:
%%writefile reducer_1.py

import sys


def reducer():
    # вид ключа: {date};{site}
    current_key = None
    current_count = 0

    for line in sys.stdin:
        key, count = line.strip().split('\t', 1)
        count = int(count)

        if key == current_key or current_key is None:
            current_count += count
        else:
            print(f"{current_key};{current_count}")
            current_count = count
            
        current_key = key

    print(f"{current_key};{current_count}")


if __name__ == "__main__":
    reducer()

Writing reducer_1.py


### 3. Запуск 1-го MapReduce таска

In [4]:
%%bashhdfs dfs -mkdir -p /hadoop/hw1/task2/mr1/input
hdfs dfs -put website_visits.csv /hadoop/hw1/task2/mr1/input/
hdfs dfs -ls /hadoop/hw1/task2/mr1/input

hdfs dfs -rm -r /hadoop/hw1/task2/mr1/output

hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-files mapper_1.py,reducer_1.py \
-mapper "python3 mapper_1.py" \
-reducer "python3 reducer_1.py" \
-input /hadoop/hw1/task2/mr1/input/ \
-output /hadoop/hw1/task2/mr1/output/

Found 1 items
-rw-r--r--   1 hadoop users   36443383 2026-09-26 21:31 /hadoop/hw1/task2/mr1/input/website_visits.csv


rm: `/hadoop/hw1/task2/mr1/output': No such file or directory


packageJobJar: [/tmp/hadoop-unjar5815108586019430255/] [] /tmp/streamjob530267466755156926.jar tmpDir=null


26/09/26 21:31:22 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.18.0.6:8032
26/09/26 21:31:22 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.18.0.6:8032
26/09/26 21:31:22 INFO JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790436132414_0008
26/09/26 21:31:23 INFO FileInputFormat: Total input files to process : 1
26/09/26 21:31:23 INFO JobSubmitter: number of splits:2
26/09/26 21:31:23 INFO JobSubmitter: Submitting tokens for job: job_1790436132414_0008
26/09/26 21:31:23 INFO JobSubmitter: Executing with tokens: []
26/09/26 21:31:23 INFO Configuration: resource-types.xml not found
26/09/26 21:31:23 INFO ResourceUtils: Unable to find 'resource-types.xml'.
26/09/26 21:31:23 INFO YarnClientImpl: Submitted application application_1790436132414_0008
26/09/26 21:31:23 INFO Job: The url to track the job: http://resourcemanager:8088/proxy/applicat

In [5]:
! hdfs dfs -ls /hadoop/hw1/task2/mr1/output

Found 2 items
-rw-r--r--   1 hadoop users          0 2026-09-26 21:31 /hadoop/hw1/task2/mr1/output/_SUCCESS
-rw-r--r--   1 hadoop users   11086384 2026-09-26 21:31 /hadoop/hw1/task2/mr1/output/part-00000


In [6]:
! hdfs dfs -getmerge /hadoop/hw1/task2/mr1/output/ ./hadoop_mr1.csv
! cat hadoop_mr1.csv

2024-05-26;http://abbott-brown.com/;1	
2024-05-26;http://abbott-ellis.com/;1	
2024-05-26;http://abbott-hall.com/;1	
2024-05-26;http://abbott-holden.com/;1	
2024-05-26;http://abbott-weaver.biz/;1	
2024-05-26;http://abbott.com/;6	
2024-05-26;http://abbott.net/;4	
2024-05-26;http://abbott.org/;5	
2024-05-26;http://acevedo-black.com/;1	
2024-05-26;http://acevedo-keith.biz/;2	
2024-05-26;http://acevedo-smith.com/;1	
2024-05-26;http://acevedo.biz/;2	
2024-05-26;http://acevedo.com/;5	
2024-05-26;http://acevedo.org/;1	
2024-05-26;http://acosta-anderson.com/;1	
2024-05-26;http://acosta-gould.net/;1	
2024-05-26;http://acosta-lloyd.com/;1	
2024-05-26;http://acosta-long.net/;1	
2024-05-26;http://acosta-martin.com/;1	
2024-05-26;http://acosta-smith.com/;1	
2024-05-26;http://acosta-spencer.com/;13	
2024-05-26;http://acosta.biz/;1	
2024-05-26;http://acosta.com/;2	
2024-05-26;http://acosta.info/;4	
2024-05-26;http://acosta.org/;2	
2024-05-26;http://adams-day.com/;2	
2024-05-26;http://adams-guerrero.co

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



После первого MapReduce получили файл hadoop_mr1.csv и сохранили его в локальную файловую систему.

### 4. Реализация шага "Map_2"

In [7]:
%%writefile mapper_2.py

import sys


def mapper():
    # на входе строка имеет следующий вид: {date};{site};{visits}\n
    for line in sys.stdin:
        date, site, visits = line.strip().split(';')
        if site != "" and date != "" and visits != "":
            print(f"{date}\t{visits}\t{site}")
        else:
            # логирование
            pass


if __name__ == "__main__":
    mapper()

Writing mapper_2.py


### 5. Реализация шага "Reduce_2"

In [8]:
%%writefile reducer_2.py

import sys


def reducer():
    # вид строки на входе: {date}\t{visits}\t{site}
    current_date = None

    for line in sys.stdin:
        date, visits, site = line.strip().split('\t')
        if date != current_date:
            current_date = date
            counter = 5
        if counter > 0:
            print(f"{date}\t{site}\t{visits}")
            counter -= 1


if __name__ == "__main__":
    reducer()

Writing reducer_2.py


### 6. Запуск 2-го MapReduce таска

После mapper_2.py данные примут вид {date}\t{visits}\t{site}, поэтому дополнительные флаги, связанные с разделительными символами, можно не прописывать.  
Опишем другие дополнительные флаги:  
-D mapreduce.job.reduces=1 - делаем только 1 Reduce, иначе придётся прописывать ещё один флаг
-D stream.num.map.output.key.fields=2 - после второго разделительного символа (по умолчанию \t) закончивается ключ и начинается значение  
-D mapreduce.partition.keycomparator.options='-k1,1 -k2,2nr' - сортировка по 1-му полю (по возрастанию), затем по 2-му полю (число, по убыванию)

In [ ]:
%%bash
hdfs dfs -mkdir -p /hadoop/hw1/task2/mr2/input
hdfs dfs -put hadoop_mr1.csv /hadoop/hw1/task2/mr2/input
hdfs dfs -ls /hadoop/hw1/task2/mr2/input
hdfs dfs -rm -r /hadoop/hw1/task2/mr2/output

hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.output.key.comparator.class=org.apache.hadoop.mapreduce.lib.partition.KeyFieldBasedComparator \
-D mapreduce.job.reduces=1 \
-D stream.num.map.output.key.fields=2 \
-D mapreduce.partition.keycomparator.options='-k1,1 -k2,2nr' \
-files mapper_2.py,reducer_2.py \
-mapper "python3 mapper_2.py" \
-reducer "python3 reducer_2.py" \
-input /hadoop/hw1/task2/mr2/input/ \
-output /hadoop/hw1/task2/mr2/output/

In [10]:
! hdfs dfs -getmerge /hadoop/hw1/task2/mr2/output/ ./hadoop_mr2.csv! cat hadoop_mr2.csv

2024-05-26	https://gonzales-bautista.com/	335
2024-05-26	http://smith.com/	235
2024-05-26	https://www.smith.com/	221
2024-05-26	http://www.smith.com/	212
2024-05-26	https://smith.com/	212
2024-05-27	https://gonzales-bautista.com/	376
2024-05-27	https://www.smith.com/	270
2024-05-27	https://smith.com/	236
2024-05-27	http://smith.com/	215
2024-05-27	http://www.smith.com/	208
2024-05-28	https://gonzales-bautista.com/	368
2024-05-28	https://smith.com/	256
2024-05-28	https://www.smith.com/	251
2024-05-28	http://smith.com/	224
2024-05-28	http://www.smith.com/	204
2024-05-29	https://gonzales-bautista.com/	402
2024-05-29	https://www.smith.com/	242
2024-05-29	http://www.smith.com/	223
2024-05-29	https://smith.com/	220
2024-05-29	http://smith.com/	206
2024-05-30	https://gonzales-bautista.com/	353
2024-05-30	https://smith.com/	246
2024-05-30	https://www.smith.com/	239
2024-05-30	http://smith.com/	229
2024-05-30	http://www.smith.com/	225
2024-05-31	https://gonzales-bautista.com/	374
2024-05-31	htt

В итоге получили то что нам нужно - топ 5 наиболее посещаемых сайтов за каждую из представленных дат.

секретная ячейка для очистки HDFS:

In [ ]:
! hdfs dfs -rm -r /hadoop/hw1/task2